# 02 — Preprocessing

In [1]:
# Re-load data

import sys
sys.path.append('..')

import pandas as pd
from config import TARGET

df = pd.read_excel('../data/raw/ObesityDataSet_raw_and_data_sinthetic.xlsx')
df.head()


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


### Train / Vault Split

In [2]:
# Split into train and vault before any encoding or scaling
# to prevent data leakage

from sklearn.model_selection import train_test_split
from config import RANDOM_STATE, TEST_SIZE, TARGET

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_vault, y_train, y_vault = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {X_train.shape}, Vault: {X_vault.shape}")
print("Vault is LOCKED — only open in the final evaluation notebook.")

Train: (1688, 16), Vault: (423, 16)
Vault is LOCKED — only open in the final evaluation notebook.


### Encode Categorical Data

In [3]:
# One-hot encode binary categorical features
# Applied to train and vault separately; vault is reindexed to match train columns

binary_features = ['Gender', 'family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']

X_train = pd.get_dummies(X_train, columns=binary_features, drop_first=True, dtype=int)
X_vault = pd.get_dummies(X_vault, columns=binary_features, drop_first=True, dtype=int)

# Align vault columns to train (handles any categories only present in vault)
X_vault = X_vault.reindex(columns=X_train.columns, fill_value=0)

X_train.head()

,Age,Height,Weight,FCVC,NCP,CAEC,CH2O,FAF,TUE,CALC,MTRANS,Gender_Male,family_history_with_overweight_yes,FAVC_yes,SMOKE_yes,SCC_yes
459,19.000000,1.760000,79.000000,2.000000,3.000000,Frequently,3.000000,1.000000,2.000000,Frequently,Public_Transportation,1,1,1,0,0
426,22.000000,1.750000,70.000000,2.000000,3.000000,Sometimes,3.000000,1.000000,1.000000,no,Public_Transportation,1,0,0,0,0
326,18.000000,1.700000,55.300000,3.000000,3.000000,Sometimes,2.000000,3.000000,0.000000,Sometimes,Public_Transportation,1,1,1,0,0
971,19.506389,1.824449,87.656029,2.793561,3.788602,Sometimes,2.429059,2.094542,0.393358,Sometimes,Public_Transportation,1,1,1,0,0
892,17.085250,1.535618,57.259124,1.972545,2.339614,Sometimes,1.711074,0.095517,1.191053,Sometimes,Public_Transportation,0,0,1,0,1


In [ ]:
# Ordinal encode CAEC and CALC
# Hardcoded order (domain knowledge) — safe to apply to both splits

caec_order = {
    'no'        : 0,
    'Sometimes' : 1,
    'Frequently': 2,
    'Always'    : 3
}

calc_order = {
    'no'        : 0,
    'Sometimes' : 1,
    'Frequently': 2,
    'Always'    : 3
}

for split in [X_train, X_vault]:
    split['CAEC'] = split['CAEC'].map(caec_order)
    split['CALC'] = split['CALC'].map(calc_order)

X_train.head()

,Age,Height,Weight,FCVC,NCP,CAEC,CH2O,FAF,TUE,CALC,MTRANS,Gender_Male,family_history_with_overweight_yes,FAVC_yes,SMOKE_yes,SCC_yes
459,19.000000,1.760000,79.000000,2.000000,3.000000,2,3.000000,1.000000,2.000000,2,Public_Transportation,1,1,1,0,0
426,22.000000,1.750000,70.000000,2.000000,3.000000,1,3.000000,1.000000,1.000000,0,Public_Transportation,1,0,0,0,0
326,18.000000,1.700000,55.300000,3.000000,3.000000,1,2.000000,3.000000,0.000000,1,Public_Transportation,1,1,1,0,0
971,19.506389,1.824449,87.656029,2.793561,3.788602,1,2.429059,2.094542,0.393358,1,Public_Transportation,1,1,1,0,0
892,17.085250,1.535618,57.259124,1.972545,2.339614,1,1.711074,0.095517,1.191053,1,Public_Transportation,0,0,1,0,1


In [5]:
# Ordinal encode MTRANS
# 0 = motorized (Automobile, Motorbike, Public_Transportation)
# 1 = physical (Bike, Walking)

mtrans_order = {
    'Public_Transportation': 0,
    'Automobile'           : 0,
    'Motorbike'            : 0,
    'Bike'                 : 1,
    'Walking'              : 1,
}

for split in [X_train, X_vault]:
    split['MTRANS'] = split['MTRANS'].map(mtrans_order)

print(X_train['MTRANS'].value_counts())

MTRANS
0    1638
1      50
Name: count, dtype: int64


In [6]:
# Ordinal encode target
# Hardcoded order based on background knowledge  

from config import TARGET_CLASSES

target_order = {label: i for i, label in enumerate(TARGET_CLASSES)}
y_train = y_train.map(target_order)
y_vault = y_vault.map(target_order)

print(y_train.value_counts().sort_index())

NObeyesdad
0    218
1    229
2    232
3    232
4    281
5    237
6    259
Name: count, dtype: int64


### Features to Remove

In [7]:
dropped_columns = ['Age', 'Height', 'Weight']
X_train.drop(columns=dropped_columns, inplace=True)
X_vault.drop(columns=dropped_columns, inplace=True)

X_train.head()

,FCVC,NCP,CAEC,CH2O,FAF,TUE,CALC,MTRANS,Gender_Male,family_history_with_overweight_yes,FAVC_yes,SMOKE_yes,SCC_yes
459,2.000000,3.000000,2,3.000000,1.000000,2.000000,2,0,1,1,1,0,0
426,2.000000,3.000000,1,3.000000,1.000000,1.000000,0,0,1,0,0,0,0
326,3.000000,3.000000,1,2.000000,3.000000,0.000000,1,0,1,1,1,0,0
971,2.793561,3.788602,1,2.429059,2.094542,0.393358,1,0,1,1,1,0,0
892,1.972545,2.339614,1,1.711074,0.095517,1.191053,1,0,0,0,1,0,1


### Feature Scaling

In [8]:
# Scale features — fit on training set only, then apply to vault

from sklearn.preprocessing import StandardScaler
from config import CONTINUOUS_FEATURES

scaler = StandardScaler()
cols_to_scale = [c for c in CONTINUOUS_FEATURES if c in X_train.columns]

X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_vault[cols_to_scale] = scaler.transform(X_vault[cols_to_scale])

X_train.head()

,FCVC,NCP,CAEC,CH2O,FAF,TUE,CALC,MTRANS,Gender_Male,family_history_with_overweight_yes,FAVC_yes,SMOKE_yes,SCC_yes
459,-0.809585,0.402353,2,1.601927,-0.024257,2.242022,2,0,1,1,1,0,0
426,-0.809585,0.402353,1,1.601927,-0.024257,0.576454,0,0,1,0,0,0,0
326,1.080202,0.402353,1,-0.029305,2.336114,-1.089115,1,0,1,1,1,0,0
971,0.690076,1.419764,1,0.670590,1.267506,-0.433950,1,0,1,1,1,0,0
892,-0.861469,-0.449641,1,-0.500610,-1.091715,0.894666,1,0,0,0,1,0,1


In [9]:
# Save processed splits to disk

import os, joblib

os.makedirs("../data/processed", exist_ok=True)

joblib.dump(X_train, "../data/processed/X_train.pkl")
joblib.dump(X_vault, "../data/processed/X_vault.pkl")
joblib.dump(y_train, "../data/processed/y_train.pkl")
joblib.dump(y_vault, "../data/processed/y_vault.pkl")

print("Splits saved to data/processed/")
print("X_vault / y_vault are LOCKED — do not load in 03_modeling.")

Splits saved to data/processed/
X_vault / y_vault are LOCKED — do not load in 03_modeling.
